# Module 5.4 — ONNX Export & Runtime

**Goal:** Export DeskMate's encoder (DeBERTa) and decoder (Qwen2.5-1.5B) to ONNX, run them with ONNX Runtime, and benchmark against the PyTorch baseline on CPU and GPU. Demonstrate I/O binding for zero-copy GPU inference.

## 1. Install Dependencies

In [ ]:
# Run once in Colab
# !pip install -q onnx onnxruntime-gpu optimum[onnxruntime] transformers rouge-score

import os, time, json
import numpy as np
from pathlib import Path
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configuration

In [ ]:
ENCODER_MODEL = 'microsoft/deberta-v3-small'
# Use your fine-tuned encoder if available:
# ENCODER_MODEL = 'models/deskmate-encoder/'

DECODER_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
# Use your merged fine-tuned decoder if available:
# DECODER_MODEL = 'models/deskmate-merged/'

ENCODER_ONNX = 'models/deskmate_encoder.onnx'
DECODER_ONNX_DIR = 'models/deskmate_decoder_onnx/'

SEQ_LEN = 128
BATCH   = 1
N_BENCH = 50    # inference repetitions for latency measurement
SMOKE_TEST = True  # set False to run real export + GPU benchmark

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)
print('Config OK')

## 3. HF Token

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
print('Token loaded:', bool(HF_TOKEN))

## 4. Load Encoder (DeBERTa)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping real encoder load.')
    enc_tokenizer = None
    enc_model     = None
else:
    enc_tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL)
    enc_model = AutoModelForSequenceClassification.from_pretrained(
        ENCODER_MODEL, num_labels=5)  # 5 intent classes for DeskMate
    enc_model.eval()
    print('Encoder loaded. Params:', sum(p.numel() for p in enc_model.parameters())//1_000_000, 'M')

## 5. Export Encoder to ONNX

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping ONNX export.')
    print('[Simulated] Encoder exported to', ENCODER_ONNX)
else:
    dummy = enc_tokenizer(
        'Example support ticket text',
        return_tensors='pt',
        padding='max_length',
        max_length=SEQ_LEN,
        truncation=True,
    )

    torch.onnx.export(
        enc_model,
        args=(dummy['input_ids'], dummy['attention_mask']),
        f=ENCODER_ONNX,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids':      {0: 'batch_size', 1: 'seq_len'},
            'attention_mask': {0: 'batch_size', 1: 'seq_len'},
            'logits':         {0: 'batch_size'},
        },
        opset_version=17,
        do_constant_folding=True,
    )
    print('Exported:', ENCODER_ONNX)
    size_mb = Path(ENCODER_ONNX).stat().st_size / 1e6
    print(f'File size: {size_mb:.1f} MB')

## 6. Validate ONNX Graph

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping ONNX validation.')
    print('[Simulated] onnx.checker.check_model: PASSED')
    print('[Simulated] Node count: 312')
else:
    import onnx
    model_proto = onnx.load(ENCODER_ONNX)
    onnx.checker.check_model(model_proto)
    print('ONNX check: PASSED')
    print(f'Nodes: {len(model_proto.graph.node)}')
    print(f'Inputs: {[i.name for i in model_proto.graph.input[:3]]}')
    print(f'Outputs: {[o.name for o in model_proto.graph.output]}')

## 7. ONNX Runtime Session — Execution Providers

In [ ]:
import onnxruntime as ort

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating ORT session setup.')
    print('[Simulated] Available providers: [CUDAExecutionProvider, CPUExecutionProvider]')
    print('[Simulated] Graph optimisation: ORT_ENABLE_ALL')
    sess_cpu = None
    sess_gpu = None
else:
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.intra_op_num_threads = os.cpu_count()

    # CPU session
    sess_cpu = ort.InferenceSession(
        ENCODER_ONNX, sess_opts, providers=['CPUExecutionProvider'])
    print('CPU session providers:', sess_cpu.get_providers())

    # GPU session (falls back to CPU if no CUDA)
    providers_gpu = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    sess_gpu = ort.InferenceSession(
        ENCODER_ONNX, sess_opts, providers=providers_gpu)
    print('GPU session providers:', sess_gpu.get_providers())

## 8. Benchmark Helpers

In [ ]:
def make_inputs(tokenizer, n=BATCH, seq=SEQ_LEN):
    text = ['customer cannot log in after password reset'] * n
    enc = tokenizer(text, padding='max_length', max_length=seq,
                    truncation=True, return_tensors='np')
    return {'input_ids': enc['input_ids'].astype(np.int64),
            'attention_mask': enc['attention_mask'].astype(np.int64)}

def bench_pytorch(model, tokenizer, n_runs=N_BENCH):
    if model is None: return None
    text = ['customer cannot log in after password reset']
    enc = tokenizer(text, padding='max_length', max_length=SEQ_LEN,
                    truncation=True, return_tensors='pt')
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model(**enc)
            times.append(time.perf_counter() - t0)
    return round(np.median(times) * 1000, 2)  # ms

def bench_ort(session, inputs, n_runs=N_BENCH):
    if session is None: return None
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(None, inputs)
        times.append(time.perf_counter() - t0)
    return round(np.median(times) * 1000, 2)  # ms

print('Helpers defined.')

## 9. CPU Latency: PyTorch vs ORT

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — using simulated CPU latency values.')
    pt_cpu_ms  = 45.2
    ort_cpu_ms = 22.8
    speedup_cpu = round(pt_cpu_ms / ort_cpu_ms, 2)
else:
    inputs_np = make_inputs(enc_tokenizer)
    pt_cpu_ms  = bench_pytorch(enc_model.cpu(), enc_tokenizer)
    ort_cpu_ms = bench_ort(sess_cpu, inputs_np)
    speedup_cpu = round(pt_cpu_ms / ort_cpu_ms, 2)

print(f'PyTorch CPU:  {pt_cpu_ms} ms')
print(f'ORT CPU:      {ort_cpu_ms} ms')
print(f'Speedup:      {speedup_cpu}x')

## 10. GPU Latency: ORT CUDA vs ORT CUDA + I/O Binding

In [ ]:
def bench_ort_iobinding(session, inputs_np, n_runs=N_BENCH):
    if session is None: return None
    # Pre-allocate GPU tensors
    io = session.io_binding()
    for name, arr in inputs_np.items():
        t = torch.from_numpy(arr).cuda().contiguous()
        io.bind_input(
            name=name, device_type='cuda', device_id=0,
            element_type=np.int64,
            shape=tuple(t.shape), buffer_ptr=t.data_ptr())
    io.bind_output('logits', device_type='cuda', device_id=0)
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        session.run_with_iobinding(io)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return round(np.median(times) * 1000, 2)

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulated GPU latency values.')
    ort_gpu_ms    = 4.1
    ort_iobind_ms = 2.0
    iobind_saving_ms = round(ort_gpu_ms - ort_iobind_ms, 2)
else:
    if not torch.cuda.is_available():
        print('No GPU — skipping GPU benchmarks.')
        ort_gpu_ms = None; ort_iobind_ms = None
    else:
        inputs_np = make_inputs(enc_tokenizer)
        ort_gpu_ms    = bench_ort(sess_gpu, inputs_np)
        ort_iobind_ms = bench_ort_iobinding(sess_gpu, inputs_np)
        iobind_saving_ms = round(ort_gpu_ms - ort_iobind_ms, 2) if ort_gpu_ms and ort_iobind_ms else None

print(f'ORT CUDA:                {ort_gpu_ms} ms')
print(f'ORT CUDA + I/O binding:  {ort_iobind_ms} ms')
print(f'I/O binding saving:      {iobind_saving_ms} ms  (CPU<->GPU copy eliminated)')

## 11. What I/O Binding Removes — Explanation

In [ ]:
explanation = '''
Without I/O binding, each ORT inference call:
  1. Copies input tensors from CPU RAM to GPU VRAM (even if they were already on GPU)
  2. Allocates a new output tensor on GPU
  3. Copies the output from GPU VRAM back to CPU RAM

With I/O binding:
  - You hand ORT a direct GPU pointer for inputs (no CPU->GPU copy)
  - You pre-allocate the output buffer on GPU (no new allocation, no GPU->CPU copy)
  - The data stays on the GPU across the entire encode->decode pipeline

Net effect: eliminates 2 memory copies per forward pass.
For DeskMate at 1000 tickets/min, this saves ~33 ms per second of serving time.
'''
print(explanation)

## 12. Export Decoder via Optimum

In [ ]:
# optimum handles the KV-cache split automatically:
# model.onnx        (prefill — processes the full prompt)
# model_with_past.onnx  (decode step — accepts past KV cache, processes one token)

if SMOKE_TEST:
    print('SMOKE_TEST=True — skipping decoder ONNX export.')
    print('[Simulated] Decoder exported to', DECODER_ONNX_DIR)
    print('[Simulated] model.onnx:           ~1.4 GB')
    print('[Simulated] model_with_past.onnx: ~1.4 GB')
    dec_tps_ort = 24.5
    dec_tps_pt  = 28.0
    dec_rouge_delta = -0.005
else:
    from optimum.onnxruntime import ORTModelForCausalLM
    from transformers import AutoTokenizer as AT

    model_ort = ORTModelForCausalLM.from_pretrained(
        DECODER_MODEL,
        export=True,
        provider='CUDAExecutionProvider' if torch.cuda.is_available() else 'CPUExecutionProvider',
        token=HF_TOKEN,
    )
    model_ort.save_pretrained(DECODER_ONNX_DIR)
    dec_tok = AT.from_pretrained(DECODER_MODEL, token=HF_TOKEN)
    dec_tok.save_pretrained(DECODER_ONNX_DIR)
    print('Decoder ONNX saved to', DECODER_ONNX_DIR)
    for f in Path(DECODER_ONNX_DIR).glob('*.onnx'):
        print(f'  {f.name}: {f.stat().st_size/1e9:.2f} GB')

## 13. Decoder Throughput: ORT vs PyTorch

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — using simulated decoder throughput.')
else:
    from optimum.onnxruntime import ORTModelForCausalLM
    from transformers import AutoTokenizer as AT

    dec_tok = AT.from_pretrained(DECODER_ONNX_DIR)
    model_ort_inf = ORTModelForCausalLM.from_pretrained(
        DECODER_ONNX_DIR,
        provider='CUDAExecutionProvider' if torch.cuda.is_available() else 'CPUExecutionProvider',
    )

    TICKET = 'I was charged twice for my subscription last month.'
    inputs = dec_tok(TICKET, return_tensors='pt')

    # ORT decoder tokens/sec
    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        model_ort_inf.generate(**inputs, max_new_tokens=100, do_sample=False)
        times.append(time.perf_counter() - t0)
    dec_tps_ort = round(100 / float(np.median(times)), 1)

print(f'Decoder ORT tokens/sec: {dec_tps_ort}')
print(f'Decoder PyTorch tokens/sec (from Module 5.2): {dec_tps_pt}')
print(f'ORT vs PyTorch: {round((dec_tps_ort - dec_tps_pt)/dec_tps_pt*100, 1)}%')

## 14. ROUGE-L Check: ORT Decoder vs PyTorch

In [ ]:
from rouge_score import rouge_scorer as rs_module

_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

GOLD = [
    {'ticket': 'I was charged twice for my subscription last month.',
     'ref': 'Contact support with your invoice numbers. We will investigate and refund within 3 business days.'},
    {'ticket': 'How do I reset my 2FA device?',
     'ref': 'Go to Account > Security > Two-Factor Authentication and click Reset Device.'},
    {'ticket': 'The CSV export button is not working.',
     'ref': 'This is a known issue ERR-500 fixed in version 4.3.0. Please update or contact support.'},
]

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulated ROUGE-L delta.')
    print(f'PyTorch ROUGE-L (mean): 0.470')
    print(f'ORT ROUGE-L (mean):     0.465')
    print(f'Delta: {dec_rouge_delta}  (gate: <= 0.01)')
    if abs(dec_rouge_delta) <= 0.01:
        print('Quality gate: PASS')
else:
    def ort_reply(ticket):
        inputs = dec_tok(f'Ticket: {ticket}', return_tensors='pt')
        out = model_ort_inf.generate(**inputs, max_new_tokens=100, do_sample=False)
        return dec_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    ort_rouges = [_scorer.score(ex['ref'], ort_reply(ex['ticket']))['rougeL'].fmeasure
                 for ex in GOLD]
    dec_rouge_delta = round(sum(ort_rouges)/len(ort_rouges) - 0.47, 4)
    print(f'ORT ROUGE-L delta vs PyTorch: {dec_rouge_delta}')
    print('Gate:', 'PASS' if abs(dec_rouge_delta) <= 0.01 else 'REVIEW')

## 15. Latency Comparison Chart

In [ ]:
import matplotlib.pyplot as plt

labels  = ['PyTorch\nCPU', 'ORT\nCPU', 'ORT\nGPU', 'ORT GPU\n+IO bind']
latency = [pt_cpu_ms, ort_cpu_ms, ort_gpu_ms, ort_iobind_ms]
colors  = ['steelblue', 'coral', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, latency, color=colors)
ax.set_ylabel('Median latency (ms) — lower is better')
ax.set_title('Encoder Inference Latency: PyTorch vs ONNX Runtime')
for bar, val in zip(bars, latency):
    if val is not None:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val} ms', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('onnx_latency_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: onnx_latency_comparison.png')

## 16. Save Report

In [ ]:
report = [
    '# ONNX Export & Runtime Report\n',
    f'Encoder model: {ENCODER_MODEL}',
    f'Decoder model: {DECODER_MODEL}',
    f'Smoke test: {SMOKE_TEST}\n',
    '## Encoder Latency (batch=1, seq=128)',
    '',
    '| Runtime | Latency (ms) | Notes |',
    '|---------|-------------|-------|',
    f'| PyTorch CPU | {pt_cpu_ms} | Baseline |',
    f'| ORT CPU (ORT_ENABLE_ALL) | {ort_cpu_ms} | ~{speedup_cpu}x speedup |',
    f'| ORT CUDA | {ort_gpu_ms} | No I/O binding |',
    f'| ORT CUDA + I/O binding | {ort_iobind_ms} | CPU<->GPU copy eliminated |',
    '',
    '## Decoder (Qwen2.5-1.5B)',
    '',
    f'| Metric | PyTorch | ORT | Delta |',
    f'|--------|---------|-----|-------|',
    f'| Tokens/sec | {dec_tps_pt} | {dec_tps_ort} | {round(dec_tps_ort - dec_tps_pt, 1)} |',
    f'| ROUGE-L delta vs PyTorch | — | {dec_rouge_delta} | gate <= 0.01 |',
    '',
    '## Checkpoint',
    '',
    '**I/O binding removes:** the CPU<->GPU memory copies for model inputs and outputs.',
    'Without it, ORT copies inputs from CPU RAM to GPU VRAM before each forward pass',
    'and copies outputs back to CPU RAM after — even when data is already on GPU.',
    'I/O binding lets you pass GPU pointers directly, eliminating both copies.',
]

with open('reports/onnx_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/onnx_report.md')
print('\n=== Module 5.4 Complete ===')
print(f'Best encoder latency: ORT GPU + I/O binding at {ort_iobind_ms} ms')
print(f'Speedup vs PyTorch CPU: {round(pt_cpu_ms / ort_iobind_ms, 1)}x')